In [1]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "notebooks") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import math
import time
from datetime import datetime
import os
import shutil # for copying checkpoint to "last"
from pathlib import Path
import gc

import torch
import torch.nn as nn
import torchvision.transforms as transform

from torch.utils.tensorboard import SummaryWriter
import matplotlib.pylab as plt
%matplotlib inline
from IPython.display import Audio

from rnencodec.utils.utils import param_breakdown
from rnencodec.utils.io import save_run_config

from rnencodec.audioDataLoader.audio_dataset import LatentDatasetConfig, EnCodecLatentDataset_constant, EnCodecLatentDataset_dynamic, latents_to_audio_simple, efficient_codes_to_latents
from torch.utils.data import DataLoader

import rnencodec.model.gru_audio_model
from rnencodec.model.gru_audio_model import GRUModelConfig

from transformers import EncodecModel

import torch.nn.functional as F

In [4]:


#p="/home/lonce/working/RNN4Codecs/output/20250926_220930_wild_WATER_CONTINUOUS"
p="/home/lonce/working/RNN4Codecs/output/20250924_144943_multiclass_test"
saved_configs = torch.load(p+"/config.pt", weights_only=False)
model_config = saved_configs["model_config"]
data_config=saved_configs["data_config"]

#params = {'sample_rate': 24000, 'runTimeStamp': '2025-09-26_22-09-28', 'datadir': '/slowdisk/data/glass-bottle-cylinder-shape-28-sounds/glassbottlefill_hf', 'filters': {}, 'savemodel': True, 'savemodel_interval': 25, 'savemodeldir': '/home/lonce/working/RNN4Codecs/output', 'num_epochs': 325, 'batches_per_epoch': 100, 'batch_size': 100, 'noise': 0.05, 'seqLen': 125, 'lr': 0.005, 'props': {'pos': None}, 'input_size': 128, 'hiddenSize': 128, 'nLayers': 3, 'inp_proportion': 6, 'cond_proportion': 1, 'codebook_size': 1024, 'dropout': 0.1, 'n_q': 8, 'quantizer_weights': [1.5385, 1.3846, 1.2308, 1.0769, 0.9231, 0.7692, 0.6154, 0.4615], 'clamp_val': 15, 'sample_mode': 'sample', 'top_n': 3, 'TF_schedule': [25, 25], 'simulate_parallel': False, 'files_per_sequence': 4}
params = {'sample_rate': 24000, 'runTimeStamp': '2025-09-24_14-49-38', 'datadir': '/slowdisk/esteban/scratchdata/syntex24/data7wav/syn7wav24_tokens_HFdataset_5', 'filters': {}, 'savemodel': True, 'savemodel_interval': 25, 'savemodeldir': '/home/lonce/working/RNN4Codecs/output', 'num_epochs': 325, 'batches_per_epoch': 100, 'batch_size': 100, 'noise': 0.05, 'seqLen': 125, 'lr': 0.005, 'props': {'c1': (0, 1), 'c2': (0, 1), 'c3': (0, 1), 'c4': (0, 1), 'c5': (0, 1), 'c6': (0, 1), 'c7': (0, 1), 'param1': (0, 1)}, 'input_size': 128, 'hiddenSize': 128, 'nLayers': 3, 'inp_proportion': 6, 'cond_proportion': 1, 'codebook_size': 1024, 'dropout': 0.1, 'n_q': 8, 'quantizer_weights': [1.5385, 1.3846, 1.2308, 1.0769, 0.9231, 0.7692, 0.6154, 0.4615], 'clamp_val': 15, 'sample_mode': 'sample', 'top_n': 3, 'TF_schedule': [25, 25], 'simulate_parallel': False, 'files_per_sequence': 4}

save_run_config(f"{p}/config_v2.pt", params=params, model_config=model_config, data_config=data_config)
print(f"wrote {p}/config_v2.pt")
params

saved to /home/lonce/working/RNN4Codecs/output/20250924_144943_multiclass_test/config_v2.pt
wrote json param file
wrote /home/lonce/working/RNN4Codecs/output/20250924_144943_multiclass_test/config_v2.pt


{'sample_rate': 24000,
 'runTimeStamp': '2025-09-24_14-49-38',
 'datadir': '/slowdisk/esteban/scratchdata/syntex24/data7wav/syn7wav24_tokens_HFdataset_5',
 'filters': {},
 'savemodel': True,
 'savemodel_interval': 25,
 'savemodeldir': '/home/lonce/working/RNN4Codecs/output',
 'num_epochs': 325,
 'batches_per_epoch': 100,
 'batch_size': 100,
 'noise': 0.05,
 'seqLen': 125,
 'lr': 0.005,
 'props': {'c1': (0, 1),
  'c2': (0, 1),
  'c3': (0, 1),
  'c4': (0, 1),
  'c5': (0, 1),
  'c6': (0, 1),
  'c7': (0, 1),
  'param1': (0, 1)},
 'input_size': 128,
 'hiddenSize': 128,
 'nLayers': 3,
 'inp_proportion': 6,
 'cond_proportion': 1,
 'codebook_size': 1024,
 'dropout': 0.1,
 'n_q': 8,
 'quantizer_weights': [1.5385,
  1.3846,
  1.2308,
  1.0769,
  0.9231,
  0.7692,
  0.6154,
  0.4615],
 'clamp_val': 15,
 'sample_mode': 'sample',
 'top_n': 3,
 'TF_schedule': [25, 25],
 'simulate_parallel': False,
 'files_per_sequence': 4}